# Sera-Z — Latih Model 1 (YOLOv11n-cls) di Colab

Notebook ini sudah final. Jalankan sel dari atas ke bawah, jangan melompat.

- Runtime: **T4 GPU** (`Runtime > Change runtime type > T4 GPU`)
- Repo private: siapkan GitHub token, atau upload `ml.zip`
- Waktu: persiapan data 10-20 menit, training 30 epoch sekitar 1,5-2 jam
- Jangan tutup tab selama training
- Hasil disimpan ke Google Drive di sel kedua dari akhir

In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada")

## 1. Ambil kode

Kalau folder repo belum ada, sel ini akan clone. Untuk repo private, tempel
GitHub token saat diminta (token tidak ditampilkan). Kalau lebih suka tanpa
token, upload `ml.zip` lalu jalankan sel unzip di bawah.

In [ ]:
import os
import subprocess

REPO = "https://github.com/Bisminee/PBL-Mobile-Kelompok2-Semester5.git"
BRANCH = "ml/model1-v2-pipeline"
DIR = "/content/sera-z"

if os.path.isdir(f"{DIR}/ml/scripts"):
    print("Folder repo sudah ada:", DIR)
elif os.path.isdir("/content/PBL-Mobile-Kelompok2-Semester5/ml/scripts"):
    DIR = "/content/PBL-Mobile-Kelompok2-Semester5"
    print("Memakai folder lama:", DIR)
else:
    from getpass import getpass
    token = getpass("GitHub token (kosongkan jika repo publik): ").strip()
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    subprocess.run(["git", "clone", "--branch", BRANCH, url, DIR], check=True)

os.chdir(DIR)
print("cwd:", os.getcwd())
print("script:", sorted(os.listdir("ml/scripts")))

Alternatif tanpa Git: upload `ml.zip` (dibuat dengan perintah di bawah), lalu
jalankan sel unzip ini.

```powershell
git archive --format=zip -o D:\Temp\ml.zip ml/model1-v2-pipeline ml
```

In [ ]:
# Jalankan hanya jika memakai ml.zip. Lewati kalau clone berhasil.
# !mkdir -p /content/sera-z
# from google.colab import files
# files.upload()
# !unzip -q ml.zip -d /content/sera-z
# import os; os.chdir("/content/sera-z")
# print(os.getcwd(), sorted(os.listdir("ml/scripts")))

## 2. Pasang dependensi

In [ ]:
!pip install -q ultralytics datasets pillow pandas scikit-learn matplotlib tqdm
import ultralytics, datasets, torch
print("ultralytics:", ultralytics.__version__)
print("datasets:", datasets.__version__)

## 3. Pastikan script COCO sudah dipatch

Patch ini membuat script membaca kategori COCO di berbagai versi `datasets`.
Aman dijalankan berulang; kalau sudah ada, tidak diubah lagi.

In [ ]:
from pathlib import Path

p = Path("ml/scripts/11_download_negatives.py")
kode = p.read_text()

if "kandidat" in kode:
    print("Patch sudah ada.")
else:
    mulai = kode.index("def nama_kategori")
    akhir = kode.index("def main()")
    fungsi_baru = '''def nama_kategori(ds, row) -> set[str] | None:
    # Nama kategori COCO satu baris. None bila tidak bisa dipetakan.
    raw = row.get("objects", {}).get("category", [])
    if not raw:
        return set()
    if all(isinstance(c, str) for c in raw):
        return set(raw)

    fitur = ds.features["objects"]
    kandidat = None
    if hasattr(fitur, "feature"):
        dalam = fitur.feature
        kandidat = dalam.get("category") if isinstance(dalam, dict) else None
    elif isinstance(fitur, dict):
        kandidat = fitur.get("category")
        if kandidat is None and isinstance(fitur.get("feature"), dict):
            kandidat = fitur["feature"].get("category")

    names = getattr(kandidat, "names", None)
    if not names:
        return None
    return {names[c] if isinstance(c, int) else c for c in raw}


'''
    kode = kode[:mulai] + fungsi_baru + kode[akhir:]
    kode = kode.replace(
        "if args.prefer_indoor and not (kategori & KATEGORI_DALAM_RUANG):",
        "if args.prefer_indoor and kategori is not None and not (kategori & KATEGORI_DALAM_RUANG):",
    )
    p.write_text(kode)
    print("Patch diterapkan.")

## 4. Unduh 800 foto objek bukan pakaian

Kelas `other` berisi laptop, meja, kursi, orang, botol, TV, buku, ruangan.
Dilewati otomatis kalau folder sudah terisi.

In [ ]:
from pathlib import Path

jumlah = len(list(Path("ml/data/raw/negatives").rglob("*.jpg")))
print("Foto negatif sekarang:", jumlah)

if jumlah < 700:
    !python ml/scripts/11_download_negatives.py --count 800
else:
    print("Sudah cukup, unduh dilewati.")

## 5. Siapkan dataset 10 kelas

Perkiraan 10-20 menit. Tanda selesai: `metadata.csv ditulis`.

In [ ]:
FORCE = True  # False = pakai dataset yang sudah ada

if FORCE:
    !python ml/scripts/09_prepare_hf.py --force --negatives ml/data/raw/negatives
else:
    print("Dilewati. Set FORCE = True untuk menyiapkan ulang.")

## 6. Verifikasi dataset

Harus ada 10 kelas termasuk `other`, dan semua split terisi.

In [ ]:
import pandas as pd

m = pd.read_csv("ml/data/processed/model1/metadata.csv")
print(m.groupby(["split", "target"]).size().unstack(fill_value=0).to_string())
print()
print("Kelas:", sorted(m["target"].unique()))
print()
print(m.groupby(["split", "source"]).size().to_string())

## 7. Latih model

30 epoch, batch 64, 224 px. Perkiraan 1,5-2 jam di T4.

In [ ]:
!python ml/scripts/04_train_yolo11n.py \
    --data ml/data/processed/model1 \
    --epochs 30 --imgsz 224 --batch 64 \
    --workers 2 --device 0 --name model1_hf_v2

## 8. Evaluasi test katalog

In [ ]:
!python ml/scripts/05_eval_model1.py \
    --data ml/data/processed/model1 \
    --weights ml/runs/model1_hf_v2/weights/best.pt

import json
d = json.load(open("ml/reports/model1_test_summary.json"))
print("Akurasi:", d["accuracy"], "| Macro F1:", d["macro_f1"])

## 9. Ekspor TFLite int8

Pastikan output verifikasi tertulis `[1, 10]`.

In [ ]:
!pip install -q tensorflow onnx onnxslim onnxruntime

!python ml/scripts/06_export_tflite.py \
    --data ml/data/processed/model1 \
    --weights ml/runs/model1_hf_v2/weights/best.pt \
    --out-name model1_yolo11n_v2.tflite

## 10. Simpan ke Google Drive

Wajib sebelum sesi Colab berakhir.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p "/content/drive/MyDrive/SeraZ-ML"
!cp -r ml/runs/model1_hf_v2 "/content/drive/MyDrive/SeraZ-ML/"
!cp ml/reports/model1_test_* "/content/drive/MyDrive/SeraZ-ML/"
!cp assets/models/model1_yolo11n_v2.tflite "/content/drive/MyDrive/SeraZ-ML/"

print("Tersimpan di /content/drive/MyDrive/SeraZ-ML")

## 11. Unduh ke laptop

In [ ]:
!zip -r -q /content/model1_v2.zip \
    ml/runs/model1_hf_v2 \
    ml/reports \
    assets/models/model1_yolo11n_v2.tflite

from google.colab import files
files.download("/content/model1_v2.zip")
files.download("/content/sera-z/assets/models/model1_yolo11n_v2.tflite")

## Setelah selesai

1. Unduh `model1_v2.zip` dan `model1_yolo11n_v2.tflite`.
2. Salin `model1_yolo11n_v2.tflite` menimpa `assets/models/yolo11n_cls.tflite`.
3. Pastikan `assets/models/labels.txt` berisi 10 kelas:

```
dress
long_pants
long_sleeve
other
outerwear
shirt
shoes
shorts
skirt
tshirt
```

4. Build ulang aplikasi:

```powershell
flutter clean
flutter pub get
flutter run
```

## Kalau sesi Colab mati

Isi `/content` hilang. Ulangi sel 1 sampai 4, lalu sel 5 dan seterusnya.
Sebelum menutup Colab, pastikan sel 10 (simpan ke Drive) sudah dijalankan.

## Catatan

- Jangan menjalankan training ulang hanya untuk melihat hasil; pakai sel 8.
- Jangan mencampur `labels.txt` 10 kelas dengan model 15 kelas, atau sebaliknya.
- Model hasil notebook ini 10 kelas. Model lama di aplikasi 15 kelas.